# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their available fields with their @id
record_sets = list(dataset.record_sets)
print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs['name'] if 'name' in rs else ''}")
    # List all fields and columns inside this record set
    if 'field' in rs:
        print("  Fields:")
        for field in rs['field']:
            print(f"    - @id: {field['@id']}, name: {field.get('name', '')}, dataType: {field.get('dataType', '')}")
        print()
    if 'column' in rs:
        print("  Columns:")
        for col in rs['column']:
            print(f"    - @id: {col['@id']}, name: {col.get('name', '')}, dataType: {col.get('dataType', '')}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Below, we load the data from all record sets detected above and display the first few rows.**

In [ ]:
# Gather all record set @ids and load their data into DataFrames
import collections
record_sets = list(dataset.record_sets)

dataframes = {}
rs_ids = [rs['@id'] for rs in record_sets]
for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {rs_id}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Select the first record set for demonstration
if rs_ids:
    first_rs = rs_ids[0]
    print(f"\nColumns in record set {first_rs}:")
    if not dataframes[first_rs].empty:
        print(dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head())
    else:
        print("No data available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

**Below, we demonstrate EDA using a numeric field and a grouping field from the chosen record set. Replace `<numeric_field_id>` and `<group_field_id>` below with appropriate column names or `@id`s from your dataset overview above.**

In [ ]:
# EDA: Filtering, normalizing, grouping by attributes
import numpy as np

# Choose the main record set for EDA
rs_id = rs_ids[0] if rs_ids else None
df = dataframes[rs_id] if rs_id else pd.DataFrame()

# Select a numeric field -- update this based on the actual columns as identified above
numeric_field = None
group_field = None
if not df.empty:
    # Try to infer numeric fields
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]  # Take the first numeric field
        print(f"Using numeric field: {numeric_field}")
    # Otherwise, try to coerce numeric columns
    else:
        # Try to find columns likely to be numeric
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_candidates:
            numeric_field = numeric_candidates[0]
            print(f"Using numeric field: {numeric_field}")
    # Attempt to select a group field (categorical)
    if numeric_field and df.shape[1]>1:
        possible_groups = [col for col in df.columns if col != numeric_field]
        group_field = possible_groups[0]
        print(f"Using group field: {group_field}")

if numeric_field and not df.empty:
    # Filter records based on threshold (set arbitrarily for demo)
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field if available
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        print(grouped_df.head())
else:
    print('No numeric field found for EDA in this record set.')

## 5. Visualization
Visualize data distributions or relationships between selected fields in the dataset.

Below, we plot histograms and group means for the numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and not df.empty:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30, alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field and group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values()
        plt.figure(figsize=(12,4))
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} per {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("No numeric field available to visualize.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we used the `mlcroissant` library to inspect and explore the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset. We loaded the Croissant schema, listed the available record sets and fields (using their `@id`s), extracted records as DataFrames, and performed basic EDA and visualizations. Further biological or statistical analysis can be performed based on the research question and the fields of interest in this dataset.*